# CoRe-AE: encode and decode a volume

The whole volume is encoded **once** into a 4 × 128³ latent (CT). The decoder runs its convolutions **once** on the latent grid, then the head renders voxels:
- **LIIF**: an MLP on the 8 neighbouring cells, chunk by chunk;
- **CLIF**: a small CNN, tile by tile with a 6-voxel halo.

The weights are downloaded from Hugging Face on first use. The same functions are available on the command line: `python inference/autoencoder.py --help`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))          # not needed after `pip install -e .`
import torch
import matplotlib.pyplot as plt
from inference.autoencoder import reconstruct, render_axial_region
from inference.common import three_planes
from c3d.hub import load_autoencoder

## 1. CT 512³

`CT_INPUT`: an AutoPET CT (`CTres.nii.gz`) or any CT NIfTI. It is resampled to 1 × 1 × 1.5 mm, cropped or padded to 512³, clipped to [−1000, 1500] HU and scaled to [−1, 1].
Set `PREPROCESSED = True` if the volume is already in [−1, 1] on that grid (the outputs below were made with a held-out AutoPET volume prepared this way).

In [ ]:
CT_INPUT = os.environ.get('C3D_DEMO_CT', 'ct.nii.gz')
MRI_INPUT = os.environ.get('C3D_DEMO_MRI', 'mri.nii.gz')
PREPROCESSED = os.environ.get('C3D_DEMO_PREPROCESSED', '0') == '1'

ct = {m: reconstruct(m, CT_INPUT, out_dir='outputs/autoencoder', preprocessed=PREPROCESSED)
      for m in ('core-ae-liif-ct512', 'core-ae-clif-ct512')}
for m, r in ct.items():
    print(f"{m}: PSNR {r['psnr']:.2f} dB  SSIM {r['ssim']:.4f}  time {r['seconds']:.1f} s  peak GPU {r['peak_gib'] or 0:.2f} GiB")

PSNR on the [−1, 1] range. SSIM here is a quick 7³-window 3-D SSIM; the paper's numbers come from its evaluation code. Times include encoding and decoding (BF16).

In [ ]:
three_planes({'input': ct['core-ae-liif-ct512']['input'],
              'LIIF': ct['core-ae-liif-ct512']['recon'],
              'CLIF': ct['core-ae-clif-ct512']['recon']}, title='CT 512³');

## 2. The decoder is continuous

LIIF predicts an intensity at **any coordinate**. Here one axial region (64 × 64 native voxels) is queried directly from the same latent on grids 1×, 2× and 4× finer, without decoding the volume again. At 1× the result equals the same region of the full decode.

In [ ]:
ae, cfg = load_autoencoder('core-ae-liif-ct512', 'cuda' if torch.cuda.is_available() else 'cpu')
z = ct['core-ae-liif-ct512']['latent'].to(next(ae.parameters()).device)
depth, y0, x0, size = 256, 280, 224, 64
crops = {s: render_axial_region(ae, z, (512, 512, 512), depth, y0, x0, size, scale=s) for s in (1, 2, 4)}
full = ct['core-ae-liif-ct512']['recon'][0, 0, depth, y0:y0 + size, x0:x0 + size]
print('max |region query - full decode| at 1x:', float((crops[1] - full).abs().max()))

fig, ax = plt.subplots(1, 4, figsize=(14, 3.8))
ax[0].imshow(ct['core-ae-liif-ct512']['input'][0, 0, depth, y0:y0 + size, x0:x0 + size], cmap='gray', vmin=-1, vmax=0.2); ax[0].set_title('input, 64²')
for a, s in zip(ax[1:], (1, 2, 4)):
    a.imshow(crops[s], cmap='gray', vmin=-1, vmax=0.2); a.set_title(f'LIIF query ×{s}, {crops[s].shape[0]}²')
for a in ax: a.axis('off')
plt.tight_layout()

Whole volumes can also be decoded on a finer grid: `reconstruct(..., scale=2)` or `--scale 2` (for CLIF too). The 2× grid has 8 times more voxels.

## 3. MRI 256³

`MRI_INPUT`: an IXI T1 volume (or any T1 NIfTI). It is cropped or padded to 256³ and normalised to [−1, 1] with the 1st–99th percentiles. The MRI models work on arrays in x, y, z order (`array_axes: xyz` in their config), so NIfTI inputs are transposed when read and written.

In [ ]:
mri = {m: reconstruct(m, MRI_INPUT, out_dir='outputs/autoencoder', preprocessed=PREPROCESSED)
       for m in ('core-ae-liif-mri256', 'core-ae-clif-mri256')}
for m, r in mri.items():
    print(f"{m}: PSNR {r['psnr']:.2f} dB  SSIM {r['ssim']:.4f}  time {r['seconds']:.1f} s  peak GPU {r['peak_gib'] or 0:.2f} GiB")
three_planes({'input': mri['core-ae-liif-mri256']['input'],
              'LIIF': mri['core-ae-liif-mri256']['recon'],
              'CLIF': mri['core-ae-clif-mri256']['recon']}, title='MRI 256³', modality='mri');

NIfTI files (CT in HU, with the 1 × 1 × 1.5 mm spacing), PNG previews and metrics are written to `outputs/autoencoder/`.